# 🌍 Example 2: Intermediate Globe (Coastline Step & Hemisphere Splitting)

In this tutorial, we will take a step forward and build a model that combines standard topography with a sharp **coastline step boundary**, and then split our globe into flat-bottomed hemispheres ready for printing.

### 🌎 Scientific & Design Context
- **The Coastline Step**: When displaying long-wavelength geophysical datasets (like seismic tomography or dynamic topography), the surface of the globe becomes distorted. Applying a sharp, raised step (e.g. 0.8 mm) at the boundaries of the continents provides a tactile, recognizable coastline anchor (like the British Isles, Madagascar, or Australia) that helps users immediately orient themselves.
- **Equatorial Splitting**: Cutting a globe along the equator and printing each half flat-side down allows you to print the spheres without any supports on the inside cavity, resulting in a clean interior surface.

## Step 1: Import Libraries

We import the core `GlobeModel` and displacer helper classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    LineDisplacer,
    calculate_displacement_scale
)

## Step 2: Generate the Base Sphere and Apply Topography

We initialize a 8,000-point Fibonacci sphere and apply ETOPO grid elevation displacement (just like in Example 1).

In [ ]:
model_radius_mm = 40.0
topo_units = 'm'  # ETOPO elevation data is in meters

model = GlobeModel(n_points=8000, radius=model_radius_mm)

# Load topography and downsample
full_grid = GeographicGrid.from_netcdf(
    "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z'
)
grid_ds = GeographicGrid(
    lats=full_grid.lats[::10],
    lons=full_grid.lons[::10],
    grid=full_grid.grid[::10, ::10]
)

scale = calculate_displacement_scale(model_radius_mm, vertical_exagg=40.0, grid_units=topo_units)
model.outer.displace(GridDisplacer(grid_ds), scale=scale)

## Step 3: Apply the Coastline Step Boundary

We load coastlines from a vector shapefile. The `LineDisplacer` identifies vertices that fall near a coastline line string and displaces them outward by an extra 0.8 mm. This creates a clean, tactile edge on the physical print.

In [ ]:
model.outer.displace(LineDisplacer(
    shapefile_path="../inputs/coastlines/ne_110m_coastline.shp",
    displacement=0.8,   # 0.8 mm step height
    width_degrees=0.5   # 0.5 degrees ribbon width
))
print("Applied coastline step displacement.")

## Step 4: Preview the Split Hemispheres in 3D

Let's split the model in half and preview the top and bottom hemispheres using a 3D scatter plot. Each hemisphere will have a flat capped bottom surface.

In [ ]:
# Generate hemispheres for preview (not hollow for this simple example)
top_half, bottom_half = model.generate_hemispheres(hollow=False)

fig = plt.figure(figsize=(12, 6))

ax1 = fig.add_subplot(121, projection='3d')
pts_top = top_half.vertices
ax1.scatter(pts_top[:, 0], pts_top[:, 1], pts_top[:, 2], c=pts_top[:, 2], cmap='viridis', s=2)
ax1.set_title("Top Hemisphere")

ax2 = fig.add_subplot(122, projection='3d')
pts_bot = bottom_half.vertices
ax2.scatter(pts_bot[:, 0], pts_bot[:, 1], pts_bot[:, 2], c=pts_bot[:, 2], cmap='viridis', s=2)
ax2.set_title("Bottom Hemisphere")

plt.show()

## Step 5: Export Split Hemispheres to STL

We use `export_hemispheres()` to perform the equatorial split and export both halves to separate STL files in a single line of code.

In [ ]:
model.export_hemispheres(
    "../outputs/example_2_top.stl",
    "../outputs/example_2_bottom.stl",
    hollow=False,
)
print("Hemispheres exported to outputs/")